# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² dataset on second primary colorectal cancer in cancer survivors using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset schema metadata is provided via a Croissant schema URL and follows the [Croissant standard](https://mlcommons.org/croissant/).

- **Schema URL**: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed. Run this once per environment.
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records using `mlcroissant`. This notebook sets the schema URL as a variable and loads the Croissant metadata object.

***Note:*** *All references use Croissant `@id` fields for precise, reproducible use across the metadata schema.*

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset title: {dataset.metadata.name}\n")
print(f"Dataset description: {dataset.metadata.description}\n")

## 2. Data Overview

Review the available record sets, fields, and their Croissant `@id`s. This helps guide later extraction and analysis steps.

Let's inspect the top-level record sets available in the dataset and display their `@id`, name, and a selection of contained fields (by their `@id`).

In [ ]:
# List all record sets and their fields by @id

record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in 'dataset.record_sets'. Trying dataset.metadata.recordSets ...")
    # Sometimes it's called 'recordSets' in metadata (plural)
    record_sets = getattr(dataset.metadata, 'recordSets', [])

if not record_sets:
    print("No record sets found. The dataset may not include any 'recordSet' entries in its schema.")
else:
    print(f"{len(record_sets)} record sets found.\n")

    for i, record_set in enumerate(record_sets):
        print(f"Record Set {i+1}:")
        print(f"  @id  : {record_set['@id']}")
        print(f"  Name : {record_set.get('name', '[name not set]')}")
        fields = record_set.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields (@id):")
        for f in fields:
            # f might be just a string (id) or a dict
            if isinstance(f, str):
                print(f"    - {f}")
            elif isinstance(f, dict):
                print(f"    - {f.get('@id', '[no id]')}")
        print()
    # We'll use the first record set as example below
    primary_record_set_id = record_sets[0]['@id']

Let's preview a few sample records from the first record set using its `@id`.

In [ ]:
# Preview some records from the first record set (using its @id)
try:
    for i, record in enumerate(dataset.records(record_set=primary_record_set_id)):
        print(record)
        if i >= 2:
            break
except Exception as e:
    print(f"Error previewing records: {e}")

## 3. Data Extraction

Load data from the record sets into pandas DataFrames using their Croissant `@id` values. This enables further processing and EDA. You should refer to the `@id` fields as shown above.

In [ ]:
# Gather all record set @id values
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load as list of records
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set: {record_set_id}")

print("\nColumns for primary record set:")
primary_df = dataframes[primary_record_set_id]
print(primary_df.columns.tolist())

print("\nFirst 5 records:")
display(primary_df.head())

## 4. Exploratory Data Analysis (EDA)

Apply some simple processing steps: filtering, normalization, grouping. We use fields present in the dataset. 

First, inspect which numeric fields are available (e.g., 'age', 'interval_between_diagnoses', or similar as per the dataset's documentation), and select one by its Croissant canonical `@id` (i.e., the column name, which is usually the field `@id`).

In [ ]:
# Inspect columns to find numeric fields
print('Data columns:')
print(primary_df.columns.tolist())

# Guess a numeric field by name
# Please update 'cr:age' to match field @id for age if available, else select another numeric field.

possible_numeric_columns = [col for col in primary_df.columns if any(keyword in col.lower() for keyword in ['age', 'interval', 'count', 'number', 'duration', 'years'])]
print(f"Possible numeric fields: {possible_numeric_columns}")

# Let's select the first matching numeric column, fallback to just the first column
if possible_numeric_columns:
    numeric_field_id = possible_numeric_columns[0]
else:
    numeric_field_id = primary_df.columns[0]  # fallback

print(f"\nUsing numeric field: {numeric_field_id}")

# Filter out non-numeric values and convert
primary_df[numeric_field_id] = pd.to_numeric(primary_df[numeric_field_id], errors='coerce')

threshold = primary_df[numeric_field_id].mean()  # Use mean as filter threshold
filtered_df = primary_df[primary_df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
display(filtered_df[[numeric_field_id]].head())

# Normalize the field
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
    filtered_df[numeric_field_id].std()
)

print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Choose a groupable field by inspection, such as 'sex', 'msi_status', 'anatomical_location', etc.
possible_categorical = [col for col in primary_df.columns if any(kw in col.lower() for kw in ['sex', 'status', 'location', 'type', 'site', 'group', 'category'])]
print(f"Possible group fields: {possible_categorical}")
if possible_categorical:
    group_field = possible_categorical[0]
    print(f"\nGrouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped means:\n", grouped_df)
else:
    group_field = None
    print("No categorical field available for grouping.")

## 5. Visualization

Let's visualize the distribution of our selected numeric field and the group means, if applicable.

In [ ]:
# Histogram of the numeric field (filtered)
plt.figure(figsize=(7,4))
filtered_df[numeric_field_id].dropna().hist(bins=10, alpha=0.7)
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.title(f'Distribution of {numeric_field_id} (filtered)')
plt.show()

# Grouped barplot if grouping field exists
if group_field is not None:
    plt.figure(figsize=(8,4))
    grouped_df.set_index(group_field)[numeric_field_id].plot(kind='bar', color='skyblue')
    plt.title(f'Average {numeric_field_id} by {group_field}')
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.xlabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, you learned how to:

- Load and inspect a FAIR²-structured dataset using the `mlcroissant` library, referencing all entities via their Croissant `@id`s.
- Load available record sets and their fields, and extract records as pandas DataFrames.
- Filter and normalize numeric fields using their canonical identifiers.
- Group and visualize data based on relevant categorical variables.

**Next steps:** Explore additional fields, conduct domain-specific analyses, or join dataframes from other record sets depending on your scientific questions.

> For more information, see the [Croissant documentation](https://mlcommons.org/croissant/) and the [`mlcroissant` GitHub repository](https://github.com/mlcommons/croissant).